# Section 2 — Cleaning, Wrangling & Preprocessing

These are our next steps after being informed by our EDA:
1. Drop null and duplicates
2. Drop irrelevant columns — nameOrig, nameDest, isFlaggedFraud
2. Engineer new features — balance delta columns that we previewed in multivariate analysis
3. Handle the balance column collinearity — oldbalanceOrg and newbalanceOrig are 0.9988 correlated
4. Encode categorical variable — 'type' needs to be converted from text to numbers for the model
5. Save as new CSV called clean_bank_transactions.csv




In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('../data/caishen_bank_transactions.csv')
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


### Drop Null and Duplicates

In [3]:
print(f"Count of Null Values: {df.isnull().sum()}")
print(f"Count of Duplicates: {df.duplicated().sum()}")

Count of Null Values: step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64
Count of Duplicates: 0


### Drop Irrelevant Columns

In [4]:
df = df.drop(columns=['nameOrig', 'nameDest', 'isFlaggedFraud'])
print(df.columns)
print(df.shape)

Index(['step', 'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig',
       'oldbalanceDest', 'newbalanceDest', 'isFraud'],
      dtype='object')
(6362620, 8)


### Data-Engineering

**Amount Change in Origin Account After Transfer**

In [5]:
# if negative: money left the account
df['balance_change_orig'] = df['newbalanceOrig'] - df['oldbalanceOrg']
df['balance_change_orig']

0            -9839.64
1            -1864.28
2             -181.00
3             -181.00
4           -11668.14
              ...    
6362615    -339682.13
6362616   -6311409.28
6362617   -6311409.28
6362618    -850002.52
6362619    -850002.52
Name: balance_change_orig, Length: 6362620, dtype: float64

**Amount Change in Destination Account After Transfer**

In [6]:
# how much changed after transfer (gain?)
df['balance_change_dest'] = df['newbalanceDest'] - df['oldbalanceDest']
df['balance_change_dest']

0                0.00
1                0.00
2                0.00
3           -21182.00
4                0.00
              ...    
6362615     339682.13
6362616          0.00
6362617    6311409.27
6362618          0.00
6362619     850002.52
Name: balance_change_dest, Length: 6362620, dtype: float64

**Difference in Amount and What was Actually Transferred**

In [7]:
# measuring how much difference there is, but should be 0 if transaction was clean
df['balance_mismatch'] = abs(df['amount'] - abs(df['balance_change_orig']))
df['balance_mismatch']

0          1.455192e-11
1          1.136868e-12
2          0.000000e+00
3          0.000000e+00
4          0.000000e+00
               ...     
6362615    0.000000e+00
6362616    0.000000e+00
6362617    0.000000e+00
6362618    0.000000e+00
6362619    0.000000e+00
Name: balance_mismatch, Length: 6362620, dtype: float64

### Handling Multi-Collinearity

From our EDA, oldbalanceOrg and newbalanceOrig had a correlation of 1.0, this is basically a redundant information. 
Because we created new features that explain the changes/discrepencies within each account, we can drop these columns to avoid multicollinearity.

In [8]:
# Drop balance columns
df = df.drop(columns=['oldbalanceOrg', 'newbalanceOrig', 
                       'oldbalanceDest', 'newbalanceDest']).copy()

print(df.columns)
print(df.shape)

Index(['step', 'type', 'amount', 'isFraud', 'balance_change_orig',
       'balance_change_dest', 'balance_mismatch'],
      dtype='object')
(6362620, 7)


### One-Hot Encoding

In [9]:
# Converting Categorical Data into Numerical - remember first column is dropped 
df = pd.get_dummies(df, columns=['type'], prefix='type', dtype=int, drop_first=True)
df.columns

Index(['step', 'amount', 'isFraud', 'balance_change_orig',
       'balance_change_dest', 'balance_mismatch', 'type_CASH_OUT',
       'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER'],
      dtype='object')

### Final Check

In [10]:
#Null Values
print(df.isnull().sum())
print(f"Number of Duplicates: {df.duplicated().sum()}")

step                   0
amount                 0
isFraud                0
balance_change_orig    0
balance_change_dest    0
balance_mismatch       0
type_CASH_OUT          0
type_DEBIT             0
type_PAYMENT           0
type_TRANSFER          0
dtype: int64
Number of Duplicates: 1156


Initially, there were no duplicated rows but because we dropped the columns (nameOrig and nameDest) that uniquely identified each row, these made some rows that had different account names but identical values in every other column became indistinguishable.

**Handling Duplicated Values:**
<br>
The attempt on dropping the duplicated rows, resulted in losing about 89 fraudulent transactions. When we were doing our EDA, we saw that only 0.13% of our dataset contained fraudulent cases, retaining every fraud instance is critical. 
<br>
Losing 89 more of these rows would cause an even more imbalance on our dataset. Although 89 cases may seem small, it's still considerable and not worth the trade off --
<br>
*so we chose to keep the duplicated rows*.

In [11]:
# we can see, because we didn't drop the duplicates, fraud count value = 1 
# or count of real fraudulent cases remain at 8213
df['isFraud'].value_counts() 

isFraud
0    6354407
1       8213
Name: count, dtype: int64

In [12]:
print(df.shape)
print(df.dtypes)

(6362620, 10)
step                     int64
amount                 float64
isFraud                  int64
balance_change_orig    float64
balance_change_dest    float64
balance_mismatch       float64
type_CASH_OUT            int64
type_DEBIT               int64
type_PAYMENT             int64
type_TRANSFER            int64
dtype: object


### Save New Dataset To New CSV File

In [16]:
df.to_csv('../data/clean_bank_transactions.csv', index=False)